# Q3 · Feature Engineering and Regression Pipeline
**Notebook:** `q3_feature_engineering.ipynb`  
**Dataset:** `../data/q3_retail_promotions.csv`  
**Goal:** Build a reproducible scikit-learn regression pipeline to predict `items_sold` at a retail store.  
**Columns:** `transaction_date`, `store_id`, `store_size`, `location_type`, `promotion_type`, `is_weekend`, `is_festival`, `competition_density`, `items_sold`  
**Author:** Gaurav Anand Shukla | BITSoM_BA_25111017

## Task 1 · Date Feature Engineering — 4 marks

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/q3_retail_promotions.csv')
print('Shape:', df.shape)
print('\nColumn Data Types:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
print('\nFirst 5 rows:')
print(df.head().to_string())

Shape: (1200, 9)

Column Data Types:
transaction_date       object
store_id                int64
store_size             object
location_type          object
promotion_type         object
is_weekend              int64
is_festival             int64
competition_density     int64
items_sold              int64
dtype: object

Missing values: 0

First 5 rows:
  transaction_date  store_id store_size location_type promotion_type  is_weekend  is_festival  competition_density  items_sold
0       2022-01-01        28      small    semi-urban      free_gift           1            0                    5         224
1       2022-01-01         5     medium    semi-urban      free_gift           1            1                    1         348
2       2022-01-02        13      small    semi-urban  loyalty_points           1            0                    6         249
3       2022-01-02        17      small         urban      free_gift           1            0                    7         259
4       2

In [2]:
# Parse transaction_date and extract date features
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

df['year']         = df['transaction_date'].dt.year
df['month']        = df['transaction_date'].dt.month
df['day_of_week']  = df['transaction_date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

print('Date features extracted successfully.')
print(f'Total rows where is_month_end=1 (day >= 25): {df["is_month_end"].sum()}')
print('\nSample output — new date columns:')
print(df[['transaction_date','year','month','day_of_week','is_month_end']].head(8).to_string())

Date features extracted successfully.
Total rows where is_month_end=1 (day >= 25): 241

Sample output — new date columns:
  transaction_date  year  month  day_of_week  is_month_end
0       2022-01-01  2022      1            5             0
1       2022-01-01  2022      1            5             0
2       2022-01-02  2022      1            6             0
3       2022-01-02  2022      1            6             0
4       2022-01-03  2022      1            0             0
5       2022-01-03  2022      1            0             0
6       2022-01-04  2022      1            1             0
7       2022-01-04  2022      1            1             0


In [3]:
# Verify is_month_end — show rows where day >= 25
sample_month_end = df[df['is_month_end']==1][['transaction_date','is_month_end']].head(5)
print('Sample rows where is_month_end = 1 (day of month >= 25):')
print(sample_month_end.to_string())
print('\nFinal dataframe shape after feature engineering:', df.shape)

Sample rows where is_month_end = 1 (day of month >= 25):
   transaction_date  is_month_end
46       2022-01-25             1
47       2022-01-25             1
48       2022-01-26             1
49       2022-01-26             1
50       2022-01-27             1

Final dataframe shape after feature engineering: (1200, 13)


## Task 2 · Temporal Train-Test Split — 3 marks

In [4]:
# Sort by date — mandatory for temporal split
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)

# Most recent 20% = test set
split_idx  = int(len(df_sorted) * 0.80)
train_df   = df_sorted.iloc[:split_idx]
test_df    = df_sorted.iloc[split_idx:]

print(f'Training set: {len(train_df)} rows')
print(f'Test set    : {len(test_df)} rows')
print(f'\nTraining date range: {train_df["transaction_date"].min().date()} → {train_df["transaction_date"].max().date()}')
print(f'Test date range    : {test_df["transaction_date"].min().date()} → {test_df["transaction_date"].max().date()}')

Training set: 960 rows
Test set    : 240 rows

Training date range: 2022-01-01 → 2024-06-11
Test date range    : 2024-06-12 → 2024-12-31


### ✅ Why a Random Split is Inappropriate for Time-Ordered Data
1. **Data leakage**: A random split allows future records (e.g., Dec 2024 sales) to appear in the training set. The model would then 'predict' data it has already seen — producing falsely optimistic metrics that collapse in production.
2. **Temporal autocorrelation**: Retail sales have strong seasonal dependencies (festival months, year-end peaks). A random split destroys these temporal structures, making the test set artificially easy.
3. **Real-world mismatch**: In production, the model always predicts future data it has never seen. The train→test direction must mirror this: **train on past (Jan 2022–Jun 2024) → predict future (Jun–Dec 2024)**.
4. **Correct approach**: Sort chronologically, use the **most recent 20%** as the test set — this is the standard for time-series and panel data evaluation.

## Task 3 · Preprocessing Pipeline — 5 marks

In [5]:
# Define feature categories
cat_features = ['promotion_type', 'location_type', 'store_size']
num_features = ['competition_density', 'is_weekend', 'is_festival',
                'month', 'day_of_week', 'is_month_end', 'year']
drop_cols    = ['transaction_date', 'store_id', 'items_sold']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['items_sold']
X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df['items_sold']

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('\nCategorical features:', cat_features)
print('Numerical  features :', num_features)

X_train shape: (960, 10)
X_test  shape: (240, 10)

Categorical features: ['promotion_type', 'location_type', 'store_size']
Numerical  features : ['competition_density', 'is_weekend', 'is_festival', 'month', 'day_of_week', 'is_month_end', 'year']


In [6]:
# Build ColumnTransformer — OHE for categoricals, StandardScaler for numericals
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features)
])

# Build complete Pipelines (preprocessing + model)
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor',    LinearRegression())
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor',    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

print('Pipelines built successfully.')
print('Linear Regression pipeline steps:', [s[0] for s in lr_pipeline.steps])
print('Random Forest pipeline steps    :', [s[0] for s in rf_pipeline.steps])

Pipelines built successfully.
Linear Regression pipeline steps: ['preprocessor', 'regressor']
Random Forest pipeline steps    : ['preprocessor', 'regressor']


In [7]:
# Fit pipelines ONLY on training data — pipeline applies transforms correctly
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
print('Both pipelines fitted on training data only.')
print('StandardScaler and OHE fitted on X_train — applied to X_test (no leakage).')

Both pipelines fitted on training data only.
StandardScaler and OHE fitted on X_train — applied to X_test (no leakage).


## Task 4 · Model Training and Evaluation — 8 marks

In [8]:
# Generate predictions on test set
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# Compute RMSE and MAE
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)

print(f'{'Model':<30} {'RMSE':>10} {'MAE':>10}')
print('-'*52)
print(f'{'Linear Regression':<30} {rmse_lr:>10.4f} {mae_lr:>10.4f}')
print(f'{'Random Forest Regressor':<30} {rmse_rf:>10.4f} {mae_rf:>10.4f}')

Model                               RMSE        MAE
----------------------------------------------------
Linear Regression                 27.1252     21.0715
Random Forest Regressor           31.2654     25.1876


In [9]:
# Parity Plots — Predicted vs Actual items_sold
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_pred, name, color, rmse, mae in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest Regressor'],
    ['#3498DB', '#E74C3C'],
    [rmse_lr, rmse_rf],
    [mae_lr, mae_rf]
):
    ax.scatter(y_test, y_pred, alpha=0.5, color=color, s=40,
               edgecolors='white', linewidth=0.3)
    lo = min(y_test.min(), y_pred.min())
    hi = max(y_test.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=2, label='Perfect Prediction')
    ax.set_title(f'{name}\nRMSE={rmse:.2f}  |  MAE={mae:.2f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual items_sold', fontsize=11)
    ax.set_ylabel('Predicted items_sold', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Parity Plots — Predicted vs Actual items_sold',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('parity_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('Parity plots saved.')

Parity plots saved.


In [10]:
# Feature Importances — Random Forest
rf_model = rf_pipeline.named_steps['regressor']
ohe      = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat']
ohe_feat = ohe.get_feature_names_out(cat_features).tolist()
all_feat = ohe_feat + num_features

importances = pd.Series(
    rf_model.feature_importances_, index=all_feat
).sort_values(ascending=False)

print('Top 10 Most Important Features (Random Forest):')
print(importances.head(10).round(4).to_string())

# Bar chart
top5 = importances.head(5)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top5.index[::-1], top5.values[::-1], color='#E74C3C', edgecolor='black')
ax.set_title('Top 5 Feature Importances — Random Forest Regressor',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (feat, val) in enumerate(zip(top5.index[::-1], top5.values[::-1])):
    ax.text(val+0.002, i, f'{val:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=100, bbox_inches='tight')
plt.show()
print('Feature importance chart saved.')

Top 10 Most Important Features (Random Forest):
is_festival               0.1763
store_size_small          0.1643
location_type_urban       0.1123
day_of_week               0.0907
is_weekend                0.0649
competition_density       0.0627
location_type_rural       0.0525
month                     0.0523
store_size_large          0.0517
promotion_type_bogo       0.0328
dtype: float64

Feature importance chart saved.


### 📊 Model Evaluation — Key Findings

| Model | RMSE | MAE |
|-------|------|-----|
| Linear Regression | **27.13** | **21.07** |
| Random Forest | 31.27 | 25.19 |

**Linear Regression outperforms Random Forest on this dataset** (lower RMSE and MAE). This is an important real-world finding — Random Forest does not always win:
- The relationship between engineered features and `items_sold` may be **sufficiently linear** for this dataset structure.
- The **temporal nature** of the split (train on 2022–2024, test on mid–late 2024) may introduce distribution shift that hurts the more complex Random Forest.
- Linear Regression's RMSE of 27.13 means predictions are on average ~27 items from actual — acceptable for store-level monthly planning.

**Top 5 Influential Features (Random Forest):**
1. **`is_festival`** (0.176) — festival periods are the single strongest driver of items sold.
2. **`store_size_small`** (0.164) — store size category is highly predictive.
3. **`location_type_urban`** (0.112) — urban stores have distinct sales patterns.
4. **`day_of_week`** (0.091) — intra-week variation in footfall.
5. **`is_weekend`** (0.065) — weekend boost in retail activity.

**Business Recommendation**: Deploy the Linear Regression pipeline (lower error, simpler, more interpretable). For future improvement, test interaction features (e.g., `is_festival × store_size`) which may close the gap.